# 04 — Explainable AI: Analisis SHAP (XAI)

**Judul Penelitian:** Eksperimen Klasifikasi Depresi pada Remaja: Perbandingan Metode Feature Selection untuk Identifikasi Fitur Gaya Hidup Paling Berpengaruh

**Tujuan notebook ini (RQ4):**
- Memverifikasi **kontribusi setiap fitur** terhadap prediksi `depression_label` menggunakan **SHAP** (Shapley Additive exPlanations) [8]
- Membandingkan hasil SHAP dengan fitur terpilih oleh **Chi-Square (FS2)** dan **Mutual Information (FS3)**
- Mengidentifikasi fitur gaya hidup dengan **kontribusi prediktif terbesar** pada model terbaik

**Catatan metodologi:**
- SHAP adalah analisis **post-hoc** — bukan metode feature selection, melainkan pelengkap interpretabilitas
- Digunakan `shap.TreeExplainer` karena model klasifikasi tetap adalah **Random Forest**
- Nilai SHAP dihitung untuk **kelas positif** (`depression_label = 1`)
- Model utama: **FS3** (Mutual Information + RF) — skenario terbaik dari notebook 03
- Analisis tambahan pada **FS0** (semua fitur) untuk perbandingan global

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
RF_PARAMS = dict(n_estimators=100, class_weight='balanced', random_state=RANDOM_STATE)

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'

for d in [FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

with open(ARTIFACTS_DIR / 'preprocessing_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)
with open(ARTIFACTS_DIR / 'experiment_fs_results.json', encoding='utf-8') as f:
    exp = json.load(f)

TARGET_COL = meta['target_col']
FEATURE_COLS = meta['feature_cols']
LIFESTYLE_FEATURES = {
    'sleep_hours', 'stress_level', 'anxiety_level', 'physical_activity',
    'daily_social_media_hours', 'screen_time_before_sleep', 'screen_time_ratio',
    'addiction_level', 'social_interaction_level', 'academic_performance',
}
FEATURE_LABELS = {
    'age': 'Usia', 'gender': 'Jenis Kelamin',
    'daily_social_media_hours': 'Durasi Media Sosial (jam/hari)',
    'sleep_hours': 'Jam Tidur', 'screen_time_before_sleep': 'Layar Sebelum Tidur (jam)',
    'academic_performance': 'Performa Akademik', 'physical_activity': 'Aktivitas Fisik',
    'social_interaction_level': 'Interaksi Sosial', 'stress_level': 'Tingkat Stres',
    'anxiety_level': 'Tingkat Kecemasan', 'addiction_level': 'Tingkat Kecanduan Digital',
    'screen_time_ratio': 'Rasio Layar/Tidur',
    'platform_Instagram': 'Platform: Instagram', 'platform_TikTok': 'Platform: TikTok',
}
print('Skenario terbaik:', exp['best_scenario'], '-', exp['best_method'])

In [ ]:
train_scaled = pd.read_csv(PROCESSED_DIR / 'train_scaled.csv')
test_scaled = pd.read_csv(PROCESSED_DIR / 'test_scaled.csv')
chi2_rank = pd.read_csv(TABLES_DIR / '13_chi2_feature_ranking.csv')
mi_rank = pd.read_csv(TABLES_DIR / '13_mi_feature_ranking.csv')
best_pipeline = joblib.load(ARTIFACTS_DIR / 'best_fs_model.joblib')

X_train = train_scaled[FEATURE_COLS]
y_train = train_scaled[TARGET_COL]
X_test = test_scaled[FEATURE_COLS]
y_test = test_scaled[TARGET_COL]
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 1. Fungsi Bantu SHAP

`TreeExplainer` menghitung Shapley values untuk setiap prediksi. Pada klasifikasi biner, kita fokus pada kontribusi fitur terhadap **kelas depresi (label=1)**.

In [ ]:
def get_positive_shap(explainer, X_df):
    sv = explainer.shap_values(X_df, check_additivity=False)
    if isinstance(sv, list):
        return sv[1]
    if getattr(sv, 'values', None) is not None and len(sv.values.shape) == 3:
        return sv.values[:, :, 1]
    if isinstance(sv, np.ndarray) and sv.ndim == 3:
        return sv[:, :, 1]
    return sv


def shap_importance_df(shap_matrix, feature_names):
    mean_abs = np.abs(shap_matrix).mean(axis=0)
    df = pd.DataFrame({
        'fitur': feature_names,
        'mean_abs_shap': mean_abs.round(6),
        'label': [FEATURE_LABELS.get(f, f) for f in feature_names],
    }).sort_values('mean_abs_shap', ascending=False)
    df['rank_shap'] = range(1, len(df) + 1)
    return df

## 2. SHAP pada Model Terbaik (FS3)

Model terbaik dari notebook 03 menggunakan pipeline **SelectKBest(MI) + Random Forest**. SHAP dihitung pada fitur yang terpilih agar interpretasi tetap pada nama fitur asli.

In [ ]:
selector = best_pipeline.named_steps['select']
rf_best = best_pipeline.named_steps['clf']
selected_features = [f for f, m in zip(FEATURE_COLS, selector.get_support()) if m]

X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)
X_test_sel_df = pd.DataFrame(X_test_sel, columns=selected_features)

explainer_fs3 = shap.TreeExplainer(rf_best)
shap_fs3 = get_positive_shap(explainer_fs3, X_test_sel_df)
imp_fs3 = shap_importance_df(shap_fs3, selected_features)
imp_fs3.to_csv(TABLES_DIR / '19_shap_importance_fs3.csv', index=False)
print('Fitur terpilih MI:', selected_features)
imp_fs3

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_fs3, X_test_sel_df, feature_names=selected_features, show=False)
plt.title(f"SHAP Summary — Model Terbaik ({exp['best_method']})", fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '19_shap_summary_fs3.png', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
shap.summary_plot(shap_fs3, X_test_sel_df, plot_type='bar', feature_names=selected_features, show=False)
plt.title('Mean |SHAP| — Model Terbaik (FS3)', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '20_shap_bar_fs3.png', bbox_inches='tight')
plt.show()

In [ ]:
top3 = imp_fs3['fitur'].head(3).tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, top3):
    idx = selected_features.index(feat)
    shap.dependence_plot(idx, shap_fs3, X_test_sel_df, feature_names=selected_features, ax=ax, show=False)
    ax.set_title(FEATURE_LABELS.get(feat, feat))
plt.suptitle('SHAP Dependence Plot — 3 Fitur Teratas (FS3)', fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '21_shap_dependence_fs3.png', bbox_inches='tight')
plt.show()

### Waterfall Plot — Contoh Prediksi Benar (True Positive)

Waterfall plot menunjukkan bagaimana setiap fitur **mendorong** prediksi dari nilai dasar (base value) menuju output akhir untuk satu individu.

In [ ]:
pred = rf_best.predict(X_test_sel)
tp_idx = np.where((y_test.values == 1) & (pred == 1))[0]
sample_idx = int(tp_idx[0]) if len(tp_idx) else 0

base_value = explainer_fs3.expected_value
if isinstance(base_value, (list, np.ndarray)):
    base_value = base_value[1] if len(base_value) > 1 else float(base_value[0])

exp_one = shap.Explanation(
    values=shap_fs3[sample_idx],
    base_values=base_value,
    data=X_test_sel_df.iloc[sample_idx].values,
    feature_names=selected_features,
)
plt.figure(figsize=(10, 6))
shap.waterfall_plot(exp_one, max_display=len(selected_features), show=False)
plt.title(f'SHAP Waterfall — Sampel True Positive (indeks {sample_idx})', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '22_shap_waterfall_sample.png', bbox_inches='tight')
plt.show()

## 3. SHAP pada Baseline (FS0) — Semua Fitur

Untuk perbandingan global RQ4, SHAP juga dihitung pada Random Forest yang dilatih dengan **seluruh 14 fitur** (tanpa seleksi). Ini memungkinkan perbandingan ranking SHAP dengan skor Chi-Square dan Mutual Information pada semua fitur.

In [ ]:
rf_fs0 = RandomForestClassifier(**RF_PARAMS)
rf_fs0.fit(X_train, y_train)
explainer_fs0 = shap.TreeExplainer(rf_fs0)
X_test_df = pd.DataFrame(X_test, columns=FEATURE_COLS)
shap_fs0 = get_positive_shap(explainer_fs0, X_test_df)
imp_fs0 = shap_importance_df(shap_fs0, FEATURE_COLS)
imp_fs0.to_csv(TABLES_DIR / '20_shap_importance_fs0.csv', index=False)
imp_fs0.head(10)

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_fs0, X_test_df, feature_names=FEATURE_COLS, show=False, max_display=14)
plt.title('SHAP Summary — Baseline Semua Fitur (FS0)', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '23_shap_summary_fs0.png', bbox_inches='tight')
plt.show()

## 4. Verifikasi Silang: SHAP vs Feature Selection (RQ4)

Membandingkan ranking fitur dari tiga sumber:
1. **Chi-Square** (filter supervised)
2. **Mutual Information** (filter supervised)
3. **SHAP** (post-hoc explainability)

Korelasi Spearman rank dan overlap top-5 fitur digunakan sebagai indikator keselarasan.

In [ ]:
comparison = pd.DataFrame({'fitur': FEATURE_COLS})
comparison = comparison.merge(chi2_rank[['fitur','chi2_score']].rename(columns={'chi2_score':'skor_chi2'}), on='fitur')
comparison = comparison.merge(mi_rank[['fitur','mi_score']].rename(columns={'mi_score':'skor_mi'}), on='fitur')
comparison = comparison.merge(
    imp_fs0[['fitur','mean_abs_shap','rank_shap']].rename(
        columns={'mean_abs_shap':'mean_abs_shap_fs0','rank_shap':'rank_shap_fs0'}),
    on='fitur')

for col in ['skor_chi2','skor_mi','mean_abs_shap_fs0']:
    mx = comparison[col].max()
    comparison[f'{col}_norm'] = comparison[col] / mx if mx > 0 else 0

comparison['terpilih_chi2'] = comparison['fitur'].isin(exp['chi2_selected_features'])
comparison['terpilih_mi'] = comparison['fitur'].isin(exp['mi_selected_features'])
comparison['overlap_fs2_fs3'] = comparison['fitur'].isin(exp['overlap_features'])
comparison['gaya_hidup'] = comparison['fitur'].isin(LIFESTYLE_FEATURES)
comparison = comparison.sort_values('mean_abs_shap_fs0', ascending=False)
comparison.to_csv(TABLES_DIR / '21_shap_fs_alignment.csv', index=False)

rho_chi2, p_chi2 = spearmanr(comparison['rank_shap_fs0'], comparison['skor_chi2'].rank(ascending=False))
rho_mi, p_mi = spearmanr(comparison['rank_shap_fs0'], comparison['skor_mi'].rank(ascending=False))

top5_shap = set(imp_fs0.head(5)['fitur'])
top5_chi2 = set(chi2_rank[chi2_rank['terpilih']].head(5)['fitur'])
top5_mi = set(mi_rank[mi_rank['terpilih']].head(5)['fitur'])
overlap_all = sorted(top5_shap & top5_chi2 & top5_mi)

rq4_summary = pd.DataFrame([
    {'indikator':'Spearman rank SHAP vs Chi-Square','nilai':round(rho_chi2,4),'p_value':round(p_chi2,4)},
    {'indikator':'Spearman rank SHAP vs Mutual Information','nilai':round(rho_mi,4),'p_value':round(p_mi,4)},
    {'indikator':'Overlap top-5 SHAP & Chi-Square','nilai':', '.join(sorted(top5_shap & top5_chi2)),'p_value':''},
    {'indikator':'Overlap top-5 SHAP & MI','nilai':', '.join(sorted(top5_shap & top5_mi)),'p_value':''},
    {'indikator':'Overlap top-5 ketiga metode','nilai':', '.join(overlap_all),'p_value':''},
])
rq4_summary.to_csv(TABLES_DIR / '22_rq4_shap_summary.csv', index=False)
rq4_summary

In [ ]:
plot_df = comparison.head(10).sort_values('mean_abs_shap_fs0_norm')
fig, ax = plt.subplots(figsize=(10, 7))
y = np.arange(len(plot_df)); h = 0.22
ax.barh(y-h, plot_df['skor_chi2_norm'], height=h, label='Chi-Square', color='#4C72B0', alpha=0.85)
ax.barh(y, plot_df['skor_mi_norm'], height=h, label='Mutual Info', color='#55A868', alpha=0.85)
ax.barh(y+h, plot_df['mean_abs_shap_fs0_norm'], height=h, label='SHAP', color='#C44E52', alpha=0.85)
ax.set_yticks(y)
ax.set_yticklabels([FEATURE_LABELS.get(f,f) for f in plot_df['fitur']])
ax.set_xlabel('Skor ternormalisasi (0-1)')
ax.set_title('RQ4: Chi-Square vs MI vs SHAP', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '24_shap_vs_feature_selection.png', bbox_inches='tight')
plt.show()

In [ ]:
rank_cols = comparison.copy()
rank_cols['rank_chi2'] = rank_cols['skor_chi2'].rank(ascending=False)
rank_cols['rank_mi'] = rank_cols['skor_mi'].rank(ascending=False)
heat = rank_cols.set_index('fitur')[['rank_shap_fs0','rank_chi2','rank_mi']].head(8)
heat.index = [FEATURE_LABELS.get(f,f) for f in heat.index]
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(heat, annot=True, fmt='.0f', cmap='YlOrRd_r', ax=ax)
ax.set_title('Ranking Fitur: SHAP vs Chi-Square vs MI')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '25_rank_heatmap_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
shap_config = {
    'model_analyzed': exp['shap_model_scenario'],
    'best_method': exp['best_method'],
    'fs3_shap_ranking': imp_fs3[['fitur','mean_abs_shap','rank_shap']].to_dict('records'),
    'fs0_top5_shap': imp_fs0.head(5)['fitur'].tolist(),
    'overlap_top5_all_methods': overlap_all,
    'spearman_shap_chi2': round(float(rho_chi2), 4),
    'spearman_shap_mi': round(float(rho_mi), 4),
}
with open(ARTIFACTS_DIR / 'shap_analysis_results.json', 'w', encoding='utf-8') as f:
    json.dump(shap_config, f, indent=2)
print('Artefak tersimpan:', ARTIFACTS_DIR / 'shap_analysis_results.json')

## Ringkasan Analisis SHAP (RQ4)

**Kontribusi fitur pada model terbaik (FS3):**
- Lihat `19_shap_importance_fs3.csv` dan grafik `19–21_shap_*.png`
- Fitur dengan mean |SHAP| tertinggi adalah yang paling memengaruhi prediksi depresi pada konteks multi-fitur

**Keselarasan dengan Feature Selection:**
- Tabel `21_shap_fs_alignment.csv` membandingkan skor Chi-Square, MI, dan SHAP
- `22_rq4_shap_summary.csv` merangkum korelasi rank dan overlap top-5 fitur
- Fitur yang konsisten di ketiga metode (Chi-Square, MI, SHAP) merupakan kandidat **paling berpengaruh** secara klinis

**Kesimpulan penelitian:**
- Pipeline eksperimen lengkap: `01_eda` → `02_preprocessing` → `03_feature_selection` → `04_shap`
- Hasil siap dirangkum ke **Laporan Hasil Eksperimen** (proposal §5.1)

**Output:**
- `results/tables/19_*.csv` s.d. `22_*.csv`
- `results/figures/19_shap_summary_fs3.png` s.d. `25_rank_heatmap_comparison.png`
- `artifacts/shap_analysis_results.json`